# Tools

## What are Tools?

**Tools** are superpowers you give to the AI. By default, AI only knows what it was trained on — it can't check today's date, search the web, or call an API.

When you give it a **tool**, the AI can call that function to get real, live information.

### How to create a tool:
```python
@tool
def my_tool(input: str) -> str:
    """Describe what this tool does — the AI reads this!"""
    return "result"
```

> **Rule:** Always write a docstring. The AI reads it to decide when and how to use the tool.

## Tool with Chat Model (`bind_tools`)

`bind_tools` tells the model **about** the tools, but the model doesn't automatically call them.
It may suggest calling a tool, but you have to handle that yourself.

> **Use an agent (next section) if you want the tool to be called automatically.**

In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.tools import tool
from datetime import datetime
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

@tool
def get_todays_date() -> str:
    """get current date and time"""
    return  datetime.now()



chat_model = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7)

tools = [get_todays_date]
chat_model_with_tool = chat_model.bind_tools(tools)

response = chat_model.invoke("What is todays date")

response.content

"Today's date is June 20, 2024."

## Tool with Agent (Recommended ✅)

An agent **automatically decides** when to call which tool. You don't need to handle tool calls manually.

In [41]:
import os
from dotenv import load_dotenv
from langchain.agents import  create_agent
from langchain.tools import tool
from datetime import datetime
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


@tool
def get_todays_date() -> str:
    """get current date and time"""
    return  datetime.now()

tools = [get_todays_date]

agent = create_agent("groq:llama-3.1-8b-instant", 
    tools=tools, 
    name = "tool testing agent",
    system_prompt="You are helpful assistant"
)



response = agent.invoke({
    "messages": [{"role": "user", "content": "What is todays date"}]
})

response["messages"][-1].content

'Please note that the actual weather and date may be different from the provided information. The provided information is based on a hypothetical scenario.'

## Tool with Schema Definition (Advanced)

For tools that take **complex inputs**, use **Pydantic** to define the input structure clearly.

**Why?**
- Tells the AI exactly what inputs your tool expects
- The AI fills in the right values automatically
- Reduces errors from wrong input types

```python
class MyInput(BaseModel):
    city: str = Field(description="Name of the city")
    units: Literal["celsius", "fahrenheit"] = Field(default="celsius")

@tool(args_schema=MyInput)
def my_tool(city: str, units: str) -> str:
    ...
```

In [ ]:
import os
from dotenv import load_dotenv
from langchain.agents import  create_agent
from langchain.tools import tool
from datetime import datetime
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

from pydantic import BaseModel, Field
from typing import Literal

class WeatherInput(BaseModel):
    """Input for weather queries."""
    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="Temperature unit preference"
    )
    include_forecast: bool = Field(
        default=False,
        description="Include 5-day forecast"
    )

@tool(args_schema=WeatherInput)
def get_weather(location: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

@tool
def get_todays_date() -> str:
    """get current date and time"""
    return  datetime.now()

tools = [get_todays_date, get_weather]

agent = create_agent("groq:llama-3.1-8b-instant", 
    tools=tools, 
    name = "tool testing agent",
    system_prompt="You are helpful assistant"
)



response = agent.invoke({
    "messages": [{"role": "user", "content": "What is weather in pune"}]
})

response["messages"][-1].content

## third part tools
https://docs.langchain.com/oss/python/integrations/tools